# Chase Eval Analytics
---

> Konstantinos Mpouros - MSc AI - University of Piraeus & NCSR "Demokritos" <br>
> Xristos Karatsalos - MSc AI - University of Piraeus & NCSR "Demokritos"

This notebook is part of the analysis suite for the chase environment experiments conducted as part of the MSc AI program. Its focus is on evaluating the performance of different agent strategies during evaluation episodes. It leverages utility functions defined in the `utils.py` module to generate insightful visualizations and metrics. Specifically, it analyzes capture rates and steps taken during evaluation episodes across various failure probabilities (`p_fail`), providing a comprehensive overview of agent performance under different conditions.

## Setup
Import libraries, paths, and helper utilities. Ensures eval results are located correctly and helpers are available for quick plots.

In [1]:

from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from utils_eval import *

NOTEBOOK_DIR = Path(__file__).parent if '__file__' in globals() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RESULTS_PATH = PROJECT_ROOT / 'results' / 'chase_eval.xlsx'
ROLL_WINDOW = 50  # eval episodes for rolling averages

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)

print(f'Notebook dir: {NOTEBOOK_DIR}')
print(f'Results file: {RESULTS_PATH}')


Notebook dir: c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook
Results file: c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\results\chase_eval.xlsx


## 1. Load data & list runs
Load all sheets (runs) from the Excel log to see what eval experiments exist and their row counts.

In [2]:
sheets = load_runs(RESULTS_PATH)
print('Sheets found:')
for name, df in sheets.items():
    print(f'  {name}: {len(df):,} rows')

Sheets found:
  fp_vs_fp: 68,239 rows
  minimaxq_selfplay: 70,555 rows
  minimaxq_vs_minimaxq: 59,440 rows
  minimaxq_vs_fp: 118,764 rows


## 2. Quick peek (first sheet)
See the first few rows to remember the columns, confirm eval-specific fields (env_seed, eval_at_episode).

In [3]:
run_name = next(iter(sheets))
sheets[run_name].head(31)


,run,p_fail,env_size,env_t_max,env_step_penalty,env_capture_reward,env_wall_penalty,env_move_reward,env_dist_reward,env_seed,episode,step,env_t,sid,sid_next,catcher_x,catcher_y,runner_x,runner_y,catcher_action_id,runner_action_id,catcher_action,runner_action,catcher_x_next,catcher_y_next,runner_x_next,runner_y_next,reward,capture,done,c_fail,r_fail,reward_runner,reward_capture_c,reward_capture_r,reward_step_c,reward_step_r,reward_wall_c,reward_wall_r,reward_move_c,reward_move_r,reward_dist_c,reward_dist_r,dist_before,dist_after,dist_delta,c_hit_wall,r_hit_wall,c_moved,r_moved,value_estimate,epsilon,phase,eval_at_episode
0,fp_vs_fp,0.1,5,30,0.01,10,0.02,0.01,0.02,10050,1,1,1,198,324,1,2,4,3,1,4,UP,RIGHT,2,2,4,4,0.00,False,False,False,False,0.01,0,0,-0.01,0,0,0.00,0.01,0.01,0.00,0.00,4,4,0,False,False,True,True,NaN,NaN,eval,50
1,fp_vs_fp,0.1,5,30,0.01,10,0.02,0.01,0.02,10050,1,2,2,324,349,2,2,4,4,4,0,RIGHT,STAY,2,3,4,4,0.02,False,False,False,False,-0.02,0,0,-0.01,0,0,0.00,0.01,0.00,0.02,-0.02,4,3,-1,False,False,True,False,NaN,NaN,eval,50
2,fp_vs_fp,0.1,5,30,0.01,10,0.02,0.01,0.02,10050,1,3,3,349,474,2,3,4,4,1,0,UP,STAY,3,3,4,4,0.02,False,False,False,False,-0.02,0,0,-0.01,0,0,0.00,0.01,0.00,0.02,-0.02,3,2,-1,False,False,True,False,NaN,NaN,eval,50
3,fp_vs_fp,0.1,5,30,0.01,10,0.02,0.01,0.02,10050,1,4,4,474,474,3,3,4,4,4,1,RIGHT,UP,3,3,4,4,-0.01,False,False,True,False,-0.02,0,0,-0.01,0,0,-0.02,0.00,0.00,0.00,0.00,2,2,0,False,True,False,False,NaN,NaN,eval,50
4,fp_vs_fp,0.1,5,30,0.01,10,0.02,0.01,0.02,10050,1,5,5,474,499,3,3,4,4,4,0,RIGHT,STAY,3,4,4,4,0.02,False,False,False,False,-0.02,0,0,-0.01,0,0,0.00,0.01,0.00,0.02,-0.02,2,1,-1,False,False,True,False,NaN,NaN,eval,50
5,fp_vs_fp,0.1,5,30,0.01,10,0.02,0.01,0.02,10050,1,6,6,499,623,3,4,4,4,1,3,UP,LEFT,4,4,4,3,0.00,False,False,False,False,0.01,0,0,-0.01,0,0,0.00,0.01,0.01,0.00,0.00,1,1,0,False,False,True,True,NaN,NaN,eval,50
6,fp_vs_fp,0.1,5,30,0.01,10,0.02,0.01,0.02,10050,1,7,7,623,597,4,4,4,3,3,3,LEFT,LEFT,4,3,4,2,0.00,False,False,False,False,0.01,0,0,-0.01,0,0,0.00,0.01,0.01,0.00,0.00,1,1,0,False,False,True,True,NaN,NaN,eval,50
7,fp_vs_fp,0.1,5,30,0.01,10,0.02,0.01,0.02,10050,1,8,8,597,571,4,3,4,2,3,3,LEFT,LEFT,4,2,4,1,0.00,False,False,False,False,0.01,0,0,-0.01,0,0,0.00,0.01,0.01,0.00,0.00,1,1,0,False,False,True,True,NaN,NaN,eval,50
8,fp_vs_fp,0.1,5,30,0.01,10,0.02,0.01,0.02,10050,1,9,9,571,570,4,2,4,1,3,3,LEFT,LEFT,4,2,4,0,-0.03,False,False,True,False,0.03,0,0,-0.01,0,0,0.00,0.00,0.01,-0.02,0.02,1,2,1,False,False,False,True,NaN,NaN,eval,50
9,fp_vs_fp,0.1,5,30,0.01,10,0.02,0.01,0.02,10050,1,10,10,570,540,4,2,4,0,3,2,LEFT,DOWN,4,1,3,0,0.00,False,False,False,False,0.01,0,0,-0.01,0,0,0.00,0.01,0.01,0.00,0.00,2,2,0,False,False,True,True,NaN,NaN,eval,50


## 3. Combine & enrich
- Attach all sheets into a single step-level DataFrame.
- Add derived columns.
- Compute episode-level metrics and rolling averages.
Produces canonical eval tables keyed by run/p_fail/env_seed for downstream analyses.

In [4]:

steps_df = add_derived_columns(combine_runs(sheets))
ep_df = episode_metrics(steps_df)
summary_seed_df = capture_rate_by_run(ep_df, include_seed=True)
summary_run_df = capture_rate_by_run(ep_df, include_seed=False)
roll_df = rolling_episode_metrics(ep_df, window=ROLL_WINDOW)

print(f'Step rows total: {len(steps_df):,}')
print(f'Episode rows total: {len(ep_df):,}')
print('Per env_seed:')
display(summary_seed_df)
print('Aggregated by run/p_fail:')
display(summary_run_df)


Step rows total: 316,998
Episode rows total: 16,000
Per env_seed:


c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:178: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return base.groupby(["run", "p_fail"], group_keys=False).apply(_add_roll)


,run,p_fail,env_seed,capture_rate,avg_steps,avg_return,eval_episode_id
0,fp_vs_fp,0.1,10050,0.8,21.7,8.047,5.5
1,fp_vs_fp,0.1,10100,0.7,20.7,7.054,15.5
2,fp_vs_fp,0.1,10150,1.0,15.5,10.051,25.5
3,fp_vs_fp,0.1,10200,0.9,17.7,9.063,35.5
4,fp_vs_fp,0.1,10250,0.8,14.9,8.050,45.5
...,...,...,...,...,...,...,...
1595,minimaxq_vs_minimaxq,0.2,49803,1.0,15.0,10.056,1955.5
1596,minimaxq_vs_minimaxq,0.2,49853,1.0,8.8,10.054,1965.5
1597,minimaxq_vs_minimaxq,0.2,49903,0.9,12.7,9.026,1975.5
1598,minimaxq_vs_minimaxq,0.2,49953,0.9,13.5,9.052,1985.5


Aggregated by run/p_fail:


,run,p_fail,capture_rate,avg_steps,avg_return,eval_episode_id
0,fp_vs_fp,0.1,0.7275,18.6070,7.321210,1000.5
1,fp_vs_fp,0.2,0.8485,15.5125,8.522380,1000.5
2,minimaxq_selfplay,0.1,0.6970,18.7055,6.986045,1000.5
3,minimaxq_selfplay,0.2,0.7750,16.5720,7.748230,1000.5
4,minimaxq_vs_fp,0.1,0.0235,29.6275,0.165560,1000.5
5,minimaxq_vs_fp,0.2,0.0160,29.7545,0.089815,1000.5
6,minimaxq_vs_minimaxq,0.1,0.8180,15.8920,8.212805,1000.5
7,minimaxq_vs_minimaxq,0.2,0.8875,13.8280,8.896390,1000.5


## 4. Evaluation history
Rolling capture rate and steps across eval episodes to track how policies behave when frozen checkpoints are tested.

* Rolling capture rate and steps per eval episode for FP self-play, to see if the heuristic converges or drifts under eval.

In [5]:

run_fp = 'fp_vs_fp'
fp_ep = ep_df.loc[ep_df['run'] == run_fp]
if fp_ep.empty:
    print('No data for FP vs FP')
else:
    display(summary_seed_df.loc[summary_seed_df['run'] == run_fp])
    plot_run_trends(fp_ep, run=run_fp, window=ROLL_WINDOW, title_prefix='FP vs FP')


,run,p_fail,env_seed,capture_rate,avg_steps,avg_return,eval_episode_id
0,fp_vs_fp,0.1,10050,0.8,21.7,8.047,5.5
1,fp_vs_fp,0.1,10100,0.7,20.7,7.054,15.5
2,fp_vs_fp,0.1,10150,1.0,15.5,10.051,25.5
3,fp_vs_fp,0.1,10200,0.9,17.7,9.063,35.5
4,fp_vs_fp,0.1,10250,0.8,14.9,8.050,45.5
...,...,...,...,...,...,...,...
395,fp_vs_fp,0.2,19800,0.9,14.1,9.041,1955.5
396,fp_vs_fp,0.2,19850,0.9,15.0,9.056,1965.5
397,fp_vs_fp,0.2,19900,0.9,19.2,9.038,1975.5
398,fp_vs_fp,0.2,19950,1.0,9.5,10.049,1985.5


c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:178: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return base.groupby(["run", "p_fail"], group_keys=False).apply(_add_roll)


* Rolling capture rate and steps per eval episode for MinimaxQ vs MinimaxQ, indicating stability of the learned equilibrium at eval time.

In [6]:

run_mm = 'minimaxq_selfplay'
mm_ep = ep_df.loc[ep_df['run'] == run_mm]
if mm_ep.empty:
    print('No data for MinimaxQ self-play')
else:
    display(summary_seed_df.loc[summary_seed_df['run'] == run_mm])
    plot_run_trends(mm_ep, run=run_mm, window=ROLL_WINDOW, title_prefix='MinimaxQ self-play')


,run,p_fail,env_seed,capture_rate,avg_steps,avg_return,eval_episode_id
400,minimaxq_selfplay,0.1,20051,0.0,30.0,-0.243,5.5
401,minimaxq_selfplay,0.1,20101,0.1,27.6,0.675,15.5
402,minimaxq_selfplay,0.1,20151,0.0,30.0,-0.359,25.5
403,minimaxq_selfplay,0.1,20201,0.5,22.1,4.776,35.5
404,minimaxq_selfplay,0.1,20251,0.1,29.5,0.763,45.5
...,...,...,...,...,...,...,...
795,minimaxq_selfplay,0.2,29801,0.9,12.2,9.040,1955.5
796,minimaxq_selfplay,0.2,29851,0.8,12.4,8.031,1965.5
797,minimaxq_selfplay,0.2,29901,1.0,13.9,10.051,1975.5
798,minimaxq_selfplay,0.2,29951,1.0,8.3,10.063,1985.5


c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:178: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



* Rolling capture rate and steps per eval episode for MinimaxQ catcher vs FP runner, showing if the RL catcher maintains an edge over the FP runner during eval.

In [7]:

run_mix = 'minimaxq_vs_fp'
mix_ep = ep_df.loc[ep_df['run'] == run_mix]
if mix_ep.empty:
    print('No data for MinimaxQ vs FP')
else:
    display(summary_seed_df.loc[summary_seed_df['run'] == run_mix])
    plot_run_trends(mix_ep, run=run_mix, window=ROLL_WINDOW, title_prefix='MinimaxQ vs FP')


,run,p_fail,env_seed,capture_rate,avg_steps,avg_return,eval_episode_id
800,minimaxq_vs_fp,0.1,30052,0.0,30.0,-0.161,5.5
801,minimaxq_vs_fp,0.1,30102,0.0,30.0,-0.103,15.5
802,minimaxq_vs_fp,0.1,30152,0.0,30.0,-0.078,25.5
803,minimaxq_vs_fp,0.1,30202,0.1,27.1,0.898,35.5
804,minimaxq_vs_fp,0.1,30252,0.0,30.0,-0.084,45.5
...,...,...,...,...,...,...,...
1195,minimaxq_vs_fp,0.2,39802,0.0,30.0,-0.080,1955.5
1196,minimaxq_vs_fp,0.2,39852,0.0,30.0,-0.042,1965.5
1197,minimaxq_vs_fp,0.2,39902,0.0,30.0,-0.049,1975.5
1198,minimaxq_vs_fp,0.2,39952,0.0,30.0,-0.061,1985.5


c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:178: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [8]:
run_mix = 'minimaxq_vs_minimaxq'
mix_ep = ep_df.loc[ep_df['run'] == run_mix]
if mix_ep.empty:
    print('No data for MinimaxQ vs MinimaxQ')
else:
    display(summary_seed_df.loc[summary_seed_df['run'] == run_mix])
    plot_run_trends(mix_ep, run=run_mix, window=ROLL_WINDOW, title_prefix='MinimaxQ vs MinimaxQ')

,run,p_fail,env_seed,capture_rate,avg_steps,avg_return,eval_episode_id
1200,minimaxq_vs_minimaxq,0.1,40053,0.1,27.3,0.737,5.5
1201,minimaxq_vs_minimaxq,0.1,40103,0.1,27.8,0.739,15.5
1202,minimaxq_vs_minimaxq,0.1,40153,0.1,27.4,0.789,25.5
1203,minimaxq_vs_minimaxq,0.1,40203,0.6,20.2,5.879,35.5
1204,minimaxq_vs_minimaxq,0.1,40253,0.5,20.4,4.849,45.5
...,...,...,...,...,...,...,...
1595,minimaxq_vs_minimaxq,0.2,49803,1.0,15.0,10.056,1955.5
1596,minimaxq_vs_minimaxq,0.2,49853,1.0,8.8,10.054,1965.5
1597,minimaxq_vs_minimaxq,0.2,49903,0.9,12.7,9.026,1975.5
1598,minimaxq_vs_minimaxq,0.2,49953,0.9,13.5,9.052,1985.5


c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:178: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



## 5. Action drift over evaluation
Rolling action mix per run/p_fail to spot eval-time policy collapse or persistent asymmetries.

In [9]:
if steps_df.empty:
    print('No steps data for action drift.')
else:
    for run in steps_df['run'].unique():
        for pf in sorted(steps_df.loc[steps_df['run']==run, 'p_fail'].dropna().unique()):
            sub = steps_df.loc[(steps_df['run']==run) & (steps_df['p_fail']==pf)]
            mix_c = rolling_action_mix(sub, who='catcher', window=ROLL_WINDOW)
            title_c = f"{run} (catcher) action drift, p_fail={pf}"
            plot_action_drift(mix_c, title=title_c)
            mix_r = rolling_action_mix(sub, who='runner', window=ROLL_WINDOW)
            title_r = f"{run} (runner) action drift, p_fail={pf}"
            plot_action_drift(mix_r, title=title_r)
    print('Action drift plots generated per run/p_fail.')


c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:199: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:212: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:199: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:212: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:199: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:212: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:199: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:212: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:199: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:212: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



...

c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:199: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:212: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:199: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:212: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:199: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:212: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:199: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:212: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



Action drift plots generated per run/p_fail.


## 6. Return quantiles over evaluation
Rolling quantiles (10/50/90%) of eval returns to capture spread, not just mean performance.

In [10]:

if ep_df.empty:
    print('No episode data for quantiles.')
else:
    quant_df = rolling_return_quantiles(ep_df, window=ROLL_WINDOW, quantiles=(0.1,0.5,0.9))
    for run in quant_df['run'].unique():
        for pf in sorted(quant_df.loc[quant_df['run']==run, 'p_fail'].dropna().unique()):
            sub = quant_df.loc[(quant_df['run']==run) & (quant_df['p_fail']==pf)]
            if sub.empty:
                continue
            fig = px.line(
                sub,
                x='eval_episode_id',
                y='value',
                color='quantile',
                title=f"{run} return quantiles (p_fail={pf})",
                labels={'eval_episode_id': 'Eval episode id', 'value': 'Return', 'quantile': 'Quantile'},
            )
            fig.update_layout(template='plotly_white')
            fig.show()
    print('Return quantile plots generated per run/p_fail.')


c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\QLearning-Chase\chase\notebook\utils_eval.py:247: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



Return quantile plots generated per run/p_fail.


## 7. State-space heatmaps
Visit density and capture locations (catcher & runner) per run and p_fail. Highlights where eval play tends to occur and where captures fire.

In [11]:
if steps_df.empty:
    print('No steps data for heatmaps.')
else:
    runs = steps_df['run'].unique()
    for run in runs:
        pfs = sorted(steps_df.loc[steps_df['run']==run, 'p_fail'].dropna().unique())
        for pf in pfs:
            plot_visit_heatmaps(steps_df, run=run, p_fail=pf)


## 8. Overall comparison across runs & p_fail
Combined view of all runs by p_fail to contrast eval capture/steps across algorithms.

In [12]:

if roll_df.empty:
    print('No data to compare.')
else:
    p_fails = sorted([pf for pf in roll_df['p_fail'].dropna().unique()])
    for pf in p_fails:
        pf_df = roll_df.loc[roll_df['p_fail'] == pf]
        if pf_df.empty:
            continue
        fig = make_subplots(
            rows=1,
            cols=2,
            shared_xaxes=True,
            subplot_titles=(f'p_fail={pf} - Rolling capture rate', f'p_fail={pf} - Rolling steps'),
        )
        for run in pf_df['run'].unique():
            sub = pf_df.loc[pf_df['run'] == run]
            fig.add_trace(
                go.Scatter(x=sub['eval_episode_id'], y=sub['roll_capture_rate'], mode='lines', name=run),
                row=1,
                col=1,
            )
            fig.add_trace(
                go.Scatter(x=sub['eval_episode_id'], y=sub['roll_steps'], mode='lines', name=run),
                row=1,
                col=2,
            )
        fig.update_yaxes(title_text='Capture rate', range=[0, 1], row=1, col=1)
        fig.update_yaxes(title_text='Steps', row=1, col=2)
        fig.update_xaxes(title_text='Eval episode id', row=1, col=1)
        fig.update_xaxes(title_text='Eval episode id', row=1, col=2)
        fig.update_layout(
            height=400,
            width=1000,
            legend_title_text=None,
            template='plotly_white',
            legend=dict(orientation='v', y=0.5, yanchor='middle', x=1.02),
            margin=dict(r=160),
        )
        fig.show()
    print('Capture/steps rolling curves plotted per p_fail. Summary tables above show averages.')


Capture/steps rolling curves plotted per p_fail. Summary tables above show averages.


## 9. Trajectory samples
Visualize example eval episodes to see how catcher/runner move. <br> 
We show shortest and longest capture episodes per run/p_fail, plus a few random captures, to understand eval behavior qualitatively.

In [13]:

if steps_df.empty:
    print('No steps data for trajectories.')
else:
    for run in steps_df['run'].unique():
        for pf in sorted(steps_df.loc[steps_df['run']==run, 'p_fail'].dropna().unique()):
            sub_caps = ep_df[(ep_df['run']==run) & (ep_df['p_fail']==pf) & ep_df['captured']]
            if sub_caps.empty:
                print(f'No captures for run={run}, p_fail={pf}')
                continue
            shortest = list(sub_caps.nsmallest(2, 'steps')[['env_seed','episode']].itertuples(index=False, name=None))
            longest = list(sub_caps.nlargest(2, 'steps')[['env_seed','episode']].itertuples(index=False, name=None))
            sampled = list(sub_caps.sample(min(3, len(sub_caps)), random_state=42)[['env_seed','episode']].itertuples(index=False, name=None))
            pairs = sorted(set(shortest + longest + sampled))
            print(f"Run={run}, p_fail={pf} eval pairs -> shortest: {shortest}, longest: {longest}, sample: {sampled}")
            traj = episode_trajectories(steps_df, run=run, p_fail=pf, pairs=pairs)
            if traj.empty:
                print('  No trajectory data loaded.')
                continue
            fig = plot_positions(traj)
            if fig:
                fig.update_layout(title=f"Trajectories – {run}, p_fail={pf} (pairs {pairs})")
    print('Trajectory samples plotted per run/p_fail using (env_seed, episode) pairs.')


Run=fp_vs_fp, p_fail=0.1 eval pairs -> shortest: [(10150, 6), (11350, 1)], longest: [(10100, 3), (10150, 4)], sample: [(13400, 4), (18550, 2), (12800, 4)]


Run=fp_vs_fp, p_fail=0.2 eval pairs -> shortest: [(10250, 1), (11200, 10)], longest: [(10050, 4), (10150, 5)], sample: [(13650, 6), (10750, 3), (10850, 5)]


Run=minimaxq_selfplay, p_fail=0.1 eval pairs -> shortest: [(20301, 8), (20351, 5)], longest: [(20651, 3), (21501, 1)], sample: [(23051, 7), (28151, 7), (27301, 9)]


Run=minimaxq_selfplay, p_fail=0.2 eval pairs -> shortest: [(20301, 6), (20451, 3)], longest: [(20701, 4), (21401, 4)], sample: [(29851, 9), (29451, 1), (23051, 5)]


Run=minimaxq_vs_minimaxq, p_fail=0.1 eval pairs -> shortest: [(40853, 3), (41003, 5)], longest: [(40353, 10), (40903, 2)], sample: [(44403, 9), (49753, 2), (46203, 2)]


Run=minimaxq_vs_minimaxq, p_fail=0.2 eval pairs -> shortest: [(40253, 5), (40253, 10)], longest: [(40503, 2), (40703, 6)], sample: [(40803, 8), (46403, 7), (46053, 7)]


Run=minimaxq_vs_fp, p_fail=0.1 eval pairs -> shortest: [(30202, 1), (30402, 6)], longest: [(30352, 3), (33902, 7)], sample: [(34502, 2), (37802, 8), (34002, 6)]


Run=minimaxq_vs_fp, p_fail=0.2 eval pairs -> shortest: [(31402, 5), (31652, 10)], longest: [(34252, 3), (35152, 2)], sample: [(38452, 2), (32702, 1), (35402, 3)]


Trajectory samples plotted per run/p_fail using (env_seed, episode) pairs.


## 10. Animated trajectory (per run)
Use `animate_episode(steps_df, run, p_fail, episode)` to view a single episode as a short animation. If `episode` is omitted, a random capture episode is chosen (or any episode if none captured). Great for micro-level eval inspection.

In [14]:
fp_caps = ep_df[(ep_df['run']=='fp_vs_fp') & (ep_df['p_fail']==0.1) & ep_df['captured']]
if fp_caps.empty:
    print('No captures found for fp_vs_fp, p_fail=0.1')
else:
    eid = int(fp_caps['eval_episode_id'].sample(1).iloc[0])
    animate_episode(steps_df, run='fp_vs_fp', p_fail=0.1, eval_episode_id=eid)

In [15]:

mm_caps = ep_df[(ep_df['run']=='minimaxq_selfplay') & (ep_df['p_fail']==0.1) & ep_df['captured']]
if mm_caps.empty:
    print('No captures found for minimaxq_selfplay, p_fail=0.1')
else:
    eid = int(mm_caps['eval_episode_id'].sample(1).iloc[0])
    animate_episode(steps_df, run='minimaxq_selfplay', p_fail=0.1, eval_episode_id=eid)


In [16]:

mix_caps = ep_df[(ep_df['run']=='minimaxq_vs_fp') & (ep_df['p_fail']==0.2) & ep_df['captured']]
if mix_caps.empty:
    print('No captures found for minimaxq_vs_fp, p_fail=0.2')
else:
    eid = int(mix_caps['eval_episode_id'].sample(1).iloc[0])
    animate_episode(steps_df, run='minimaxq_vs_fp', p_fail=0.2, eval_episode_id=eid)


In [17]:
mix_caps = ep_df[(ep_df['run']=='minimaxq_vs_minimaxq') & (ep_df['p_fail']==0.2) & ep_df['captured']]
if mix_caps.empty:
    print('No captures found for minimaxq_vs_minimaxq, p_fail=0.2')
else:
    eid = int(mix_caps['eval_episode_id'].sample(1).iloc[0])
    animate_episode(steps_df, run='minimaxq_vs_minimaxq', p_fail=0.2, eval_episode_id=eid)


## 11. Outcome histogram
Steps to capture only (evaluation episodes with capture), split by run/p_fail, to gauge how quickly agents finish eval games and where long tails remain.

In [18]:

if ep_df.empty:
    print('No episode data for outcome histogram.')
else:
    caps = ep_df[ep_df['captured']]
    if caps.empty:
        print('No capture episodes to plot.')
    else:
        fig = px.histogram(
            caps,
            x='steps',
            facet_row='p_fail',
            facet_col='run',
            nbins=30,
            title='Steps to capture (evaluation episodes)',
            labels={'steps': 'Steps to capture', 'count': 'Count'},
            opacity=0.9,
        )
        fig.update_layout(template='plotly_white')
        fig.show()


## 12. Distance decay
Opening manhattan distance (step 1) per eval episode, smoothed; reveals how often eval starts are already close/far and if seeding creates easier/harder positions over time.

In [19]:

if steps_df.empty:
    print('No steps data for distance decay.')
else:
    openings = steps_df.loc[steps_df['step'] == 1].copy()
    openings = openings.groupby(['run', 'p_fail', 'env_seed', 'eval_episode_id'], as_index=False)['manhattan_dist'].mean()
    openings = openings.sort_values('eval_episode_id')
    openings['roll_dist'] = openings.groupby(['run', 'p_fail'])['manhattan_dist'].transform(lambda s: s.rolling(window=ROLL_WINDOW, min_periods=5).mean())
    fig = px.line(
        openings,
        x='eval_episode_id',
        y='roll_dist',
        color='run',
        facet_row='p_fail',
        title='Opening manhattan distance (rolling) vs eval episode id',
        labels={'roll_dist': 'Opening distance<br>(rolling)', 'eval_episode_id': 'Eval episode id'},
    )
    fig.update_layout(template='plotly_white')
    fig.update_layout(template='plotly_white')
    fig.show()


## 13. Failure-prob effect
Average move-failure rate per agent over eval episodes (rolling-smoothed) to see resilience to action failures during eval and whether failures cluster.

In [20]:

if steps_df.empty:
    print('No steps data for failure-prob effect.')
else:
    fails = steps_df.groupby(['run', 'p_fail', 'env_seed', 'eval_episode_id'], as_index=False)[['c_fail', 'r_fail']].mean()
    fails = fails.melt(id_vars=['run', 'p_fail', 'env_seed', 'eval_episode_id'], var_name='agent', value_name='fail_rate')
    fails = fails.sort_values('eval_episode_id')
    fails['roll_fail'] = fails.groupby(['run', 'p_fail', 'agent'])['fail_rate'].transform(lambda s: s.rolling(window=ROLL_WINDOW, min_periods=5).mean())
    fig = px.line(
        fails,
        x='eval_episode_id',
        y='roll_fail',
        color='agent',
        facet_col='run',
        facet_row='p_fail',
        title='Move failure rate (rolling) vs eval episode id',
        labels={'eval_episode_id': 'Eval episode id', 'roll_fail': 'Failure rate<br>(rolling)', 'agent': 'Agent'},
    )
    fig.update_layout(template='plotly_white')
    fig.update_layout(template='plotly_white')
    fig.show()


## 14. Seeds variability
Capture rate and steps per env_seed to spot sensitivity/variance across eval seeds for each run/p_fail, highlighting brittle or robust policies.

In [21]:

if ep_df.empty or 'env_seed' not in ep_df:
    print('No env_seed data for seeds variability.')
else:
    seed_summary = ep_df.groupby(['run','p_fail','env_seed'], as_index=False).agg(
        capture_rate=('captured','mean'),
        avg_steps=('steps','mean'),
    )
    fig1 = px.bar(
        seed_summary,
        x='env_seed', y='capture_rate', color='run',
        facet_row='p_fail',
        title='Capture rate per env_seed',
        labels={'env_seed':'Env seed','capture_rate':'Capture rate'},
    )
    fig1.update_layout(template='plotly_white')
    fig1.show()

    fig2 = px.box(
        ep_df,
        x='env_seed', y='steps', color='run',
        facet_row='p_fail',
        title='Steps distribution per env_seed',
        labels={'env_seed':'Env seed','steps':'Steps'},
    )
    fig2.update_layout(template='plotly_white')
    fig2.show()


## 15. Reward Shaping Breakdown
Decompose total eval rewards into components (catch, distance, time, failure penalties) per agent over eval episodes (rolling-smoothed). Shows which reward terms drive performance during evaluation.

In [22]:
X_COL = "eval_at_episode" if "eval_at_episode" in steps_df.columns else "eval_episode_id"
SMOOTH_WINDOW = 3

components = {
    "capture": ("reward_capture_c", "reward_capture_r"),
    "step": ("reward_step_c", "reward_step_r"),
    "wall": ("reward_wall_c", "reward_wall_r"),
    "move": ("reward_move_c", "reward_move_r"),
    "dist": ("reward_dist_c", "reward_dist_r"),
}

for run in steps_df["run"].unique():
    p_fails = sorted(steps_df.loc[steps_df["run"] == run, "p_fail"].dropna().unique())
    for p_fail in p_fails:
        rows = []
        for name, (c_col, r_col) in components.items():
            if c_col not in steps_df.columns or r_col not in steps_df.columns:
                print(f"Missing: {c_col} or {r_col}")
                continue

            g_c = steps_df.groupby(["run", "p_fail", X_COL], as_index=False)[c_col].mean()
            g_c = g_c.rename(columns={c_col: "value"})
            g_c["component"] = name
            g_c["who"] = "catcher"

            g_r = steps_df.groupby(["run", "p_fail", X_COL], as_index=False)[r_col].mean()
            g_r = g_r.rename(columns={r_col: "value"})
            g_r["component"] = name
            g_r["who"] = "runner"

            rows.append(g_c)
            rows.append(g_r)

        if not rows:
            continue
        long = pd.concat(rows, ignore_index=True)
        sub = long[(long["run"] == run) & (long["p_fail"] == p_fail)]
        if sub.empty:
            continue

        sub["value_smooth"] = (
            sub.sort_values(X_COL)
                .groupby(["who", "component"])["value"]
                .transform(lambda s: s.rolling(SMOOTH_WINDOW, min_periods=1).mean())
        )

        fig = px.line(
            sub,
            x=X_COL,
            y="value_smooth",
            color="who",
            facet_col="component",
            facet_col_wrap=3,
            title=f"Eval reward components (smoothed) — run={run}, p_fail={p_fail}",
            labels={"value_smooth": "Mean reward per step", "who": "agent"},
        )
        fig.update_layout(template="plotly_white")
        fig.show()

C:\Users\user\AppData\Local\Temp\ipykernel_19592\2943620556.py:41: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



C:\Users\user\AppData\Local\Temp\ipykernel_19592\2943620556.py:41: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



C:\Users\user\AppData\Local\Temp\ipykernel_19592\2943620556.py:41: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



C:\Users\user\AppData\Local\Temp\ipykernel_19592\2943620556.py:41: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



C:\Users\user\AppData\Local\Temp\ipykernel_19592\2943620556.py:41: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



C:\Users\user\AppData\Local\Temp\ipykernel_19592\2943620556.py:41: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



C:\Users\user\AppData\Local\Temp\ipykernel_19592\2943620556.py:41: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



C:\Users\user\AppData\Local\Temp\ipykernel_19592\2943620556.py:41: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



## 16. Wall Hits & Movement Rates
Track wall collisions and effective movement rates per agent over eval episodes (rolling-smoothed). Indicates navigation efficiency and potential issues during evaluation.

In [23]:
WINDOW = 50

for run in steps_df["run"].unique():
    p_fails = sorted(steps_df.loc[steps_df["run"] == run, "p_fail"].dropna().unique())
    for p_fail in p_fails:
        sub = steps_df[(steps_df["run"] == run) & (steps_df["p_fail"] == p_fail)]
        if sub.empty:
            continue

        X_COL = "eval_episode_id" if "eval_episode_id" in sub.columns else "episode"
        rates = sub.groupby(X_COL, as_index=False).agg(
            c_hit_wall=("c_hit_wall", "mean"),
            c_moved=("c_moved", "mean"),
            r_hit_wall=("r_hit_wall", "mean"),
            r_moved=("r_moved", "mean"),
        )
        rates["c_other"] = 1 - rates["c_hit_wall"] - rates["c_moved"]
        rates["r_other"] = 1 - rates["r_hit_wall"] - rates["r_moved"]

        for col in ["c_hit_wall", "c_moved", "c_other", "r_hit_wall", "r_moved", "r_other"]:
            rates[f"roll_{col}"] = rates[col].rolling(WINDOW, min_periods=5).mean()

        fig = make_subplots(
            rows=1,
            cols=2,
            subplot_titles=("Catcher", "Runner"),
            shared_yaxes=True,
        )

        colors = {
            "moved": "#f8be00",
            "hit_wall": "#d62728",
            "other": "#0094f7",
        }

        fig.add_trace(
            go.Scatter(
                x=rates[X_COL], y=rates["roll_c_moved"], name="moved",
                legendgroup="moved", stackgroup="catcher", line=dict(color=colors["moved"])
            ),
            row=1, col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=rates[X_COL], y=rates["roll_c_hit_wall"], name="hit_wall",
                legendgroup="hit_wall", stackgroup="catcher", line=dict(color=colors["hit_wall"])
            ),
            row=1, col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=rates[X_COL], y=rates["roll_c_other"], name="other (stay/fail)",
                legendgroup="other", stackgroup="catcher", line=dict(color=colors["other"])
            ),
            row=1, col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=rates[X_COL], y=rates["roll_r_moved"], name="moved",
                legendgroup="moved", stackgroup="runner", line=dict(color=colors["moved"]),
                showlegend=False
            ),
            row=1, col=2,
        )
        fig.add_trace(
            go.Scatter(
                x=rates[X_COL], y=rates["roll_r_hit_wall"], name="hit_wall",
                legendgroup="hit_wall", stackgroup="runner", line=dict(color=colors["hit_wall"]),
                showlegend=False
            ),
            row=1, col=2,
        )
        fig.add_trace(
            go.Scatter(
                x=rates[X_COL], y=rates["roll_r_other"], name="other (stay/fail)",
                legendgroup="other", stackgroup="runner", line=dict(color=colors["other"]),
                showlegend=False
            ),
            row=1, col=2,
        )

        fig.update_yaxes(range=[0, 1], title_text="Rate", row=1, col=1)
        fig.update_yaxes(range=[0, 1], title_text="Rate", row=1, col=2)
        fig.update_xaxes(title_text=X_COL, row=1, col=1)
        fig.update_xaxes(title_text=X_COL, row=1, col=2)
        fig.update_layout(
            title=f"Eval wall-hit/move/other rates (run={run}, p_fail={p_fail})",
            template="plotly_white",
            height=450,
            width=1100,
        )
        fig.show()

## 17. Distance Change Mix
Rolling mix of distance change categories (closer, same, farther) per agent over eval episodes. Reveals how agents' actions impact relative positioning during evaluation.

In [24]:
import numpy as np

WINDOW = 50

for run in steps_df["run"].unique():
    p_fails = sorted(steps_df.loc[steps_df["run"] == run, "p_fail"].dropna().unique())
    for p_fail in p_fails:
        if "dist_delta" not in steps_df.columns:
            print("Missing column: dist_delta")
            continue

        tmp = steps_df[(steps_df["run"] == run) & (steps_df["p_fail"] == p_fail)].copy()
        if tmp.empty:
            continue

        X_COL = "eval_episode_id" if "eval_episode_id" in tmp.columns else "episode"
        tmp["dist_change"] = np.sign(tmp["dist_delta"]).map({-1: "closer", 0: "same", 1: "farther"})

        mix = (
            tmp.groupby([X_COL, "dist_change"])
            .size()
            .reset_index(name="count")
            .pivot(index=X_COL, columns="dist_change", values="count")
            .fillna(0)
        )
        frac = mix.div(mix.sum(axis=1), axis=0)
        roll = frac.rolling(WINDOW, min_periods=5).mean().reset_index()
        long = roll.melt(id_vars=X_COL, var_name="dist_change", value_name="frac")

        fig = px.area(
            long,
            x=X_COL,
            y="frac",
            color="dist_change",
            title=f"Eval distance change mix (rolling) — run={run}, p_fail={p_fail}",
            labels={"frac": "Fraction of steps"},
        )
        fig.update_yaxes(range=[0, 1])
        fig.update_layout(template="plotly_white")
        fig.show()